In [1]:
import math
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()

In [2]:
dokumen = [
    "Sistem komputer membutuhkan sistem operasi untuk mengelola sumber daya perangkat keras.",
    "Jaringan komputer menghubungkan perangkat untuk berbagi data dan komunikasi.",
    "Kecerdasan buatan memanfaatkan algoritma pembelajaran mesin untuk mengolah data.",
    "Sistem temu kembali informasi mengindeks dokumen teks untuk pencarian data cepat."
]

for i, teks in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {teks}")

Dokumen 1: Sistem komputer membutuhkan sistem operasi untuk mengelola sumber daya perangkat keras.
Dokumen 2: Jaringan komputer menghubungkan perangkat untuk berbagi data dan komunikasi.
Dokumen 3: Kecerdasan buatan memanfaatkan algoritma pembelajaran mesin untuk mengolah data.
Dokumen 4: Sistem temu kembali informasi mengindeks dokumen teks untuk pencarian data cepat.


In [3]:
def preprocess_sederhana(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-zA-Z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    tokens = teks.split()
    tokens = [t for t in tokens if t not in stopwords]
    return tokens

dokumen_bersih = []
for doc in dokumen:
    tokens = preprocess_sederhana(doc)
    dokumen_bersih.append(tokens)

for i, tokens in enumerate(dokumen_bersih, 1):
    print(f"D{i}: {tokens}")

D1: ['sistem', 'komputer', 'membutuhkan', 'sistem', 'operasi', 'mengelola', 'sumber', 'daya', 'perangkat', 'keras']
D2: ['jaringan', 'komputer', 'menghubungkan', 'perangkat', 'berbagi', 'data', 'komunikasi']
D3: ['kecerdasan', 'buatan', 'memanfaatkan', 'algoritma', 'pembelajaran', 'mesin', 'mengolah', 'data']
D4: ['sistem', 'temu', 'informasi', 'mengindeks', 'dokumen', 'teks', 'pencarian', 'data', 'cepat']


In [4]:
vocab = sorted(list(set([kata for doc in dokumen_bersih for kata in doc])))

bow_list = []
for doc in dokumen_bersih:
    baris_bow = {}
    for kata in vocab:
        baris_bow[kata] = doc.count(kata)
    bow_list.append(baris_bow)

df_bow = pd.DataFrame(bow_list, index=[f"D{i}" for i in range(1, len(dokumen) + 1)])
print("=" * 70)
print("TABEL BAG-OF-WORDS (RAW COUNT)")
print("=" * 70)
print(df_bow)

TABEL BAG-OF-WORDS (RAW COUNT)
    algoritma  berbagi  buatan  cepat  data  daya  dokumen  informasi  \
D1          0        0       0      0     0     1        0          0   
D2          0        1       0      0     1     0        0          0   
D3          1        0       1      0     1     0        0          0   
D4          0        0       0      1     1     0        1          1   

    jaringan  kecerdasan  ...  mengolah  mesin  operasi  pembelajaran  \
D1         0           0  ...         0      0        1             0   
D2         1           0  ...         0      0        0             0   
D3         0           1  ...         1      1        0             1   
D4         0           0  ...         0      0        0             0   

    pencarian  perangkat  sistem  sumber  teks  temu  
D1          0          1       2       1     0     0  
D2          0          1       0       0     0     0  
D3          0          0       0       0     0     0  
D4          1    

In [5]:
tf_list = []
for doc in dokumen_bersih:
    total_kata = len(doc)
    baris_tf = {}
    for kata in vocab:
        baris_tf[kata] = doc.count(kata) / total_kata
    tf_list.append(baris_tf)

df_tf = pd.DataFrame(tf_list, index=[f"D{i}" for i in range(1, len(dokumen) + 1)])
print("=" * 70)
print("TABEL TERM FREQUENCY (TF) MANUAL")
print("=" * 70)
print(df_tf.round(3))

TABEL TERM FREQUENCY (TF) MANUAL
    algoritma  berbagi  buatan  cepat   data  daya  dokumen  informasi  \
D1      0.000    0.000   0.000  0.000  0.000   0.1    0.000      0.000   
D2      0.000    0.143   0.000  0.000  0.143   0.0    0.000      0.000   
D3      0.125    0.000   0.125  0.000  0.125   0.0    0.000      0.000   
D4      0.000    0.000   0.000  0.111  0.111   0.0    0.111      0.111   

    jaringan  kecerdasan  ...  mengolah  mesin  operasi  pembelajaran  \
D1     0.000       0.000  ...     0.000  0.000      0.1         0.000   
D2     0.143       0.000  ...     0.000  0.000      0.0         0.000   
D3     0.000       0.125  ...     0.125  0.125      0.0         0.125   
D4     0.000       0.000  ...     0.000  0.000      0.0         0.000   

    pencarian  perangkat  sistem  sumber   teks   temu  
D1      0.000      0.100   0.200     0.1  0.000  0.000  
D2      0.000      0.143   0.000     0.0  0.000  0.000  
D3      0.000      0.000   0.000     0.0  0.000  0.000  
D4

In [6]:
# Menghitung Document Frequency (DF) dan Inverse Document Frequency (IDF) Manual
# Rumus IDF: log10(N / DF) + 1
N = len(dokumen)
df_dict = {}
idf_dict = {}

for kata in vocab:
    jumlah_dokumen = sum(1 for doc in dokumen_bersih if kata in doc)
    df_dict[kata] = jumlah_dokumen
    idf_dict[kata] = math.log10(N / jumlah_dokumen) + 1.0

df_idf = pd.DataFrame({
    "DF": df_dict,
    "IDF": idf_dict
}).T

print("=" * 70)
print("TABEL DOCUMENT FREQUENCY (DF) DAN IDF MANUAL")
print("=" * 70)
print(df_idf.round(4))

TABEL DOCUMENT FREQUENCY (DF) DAN IDF MANUAL
     algoritma  berbagi  buatan   cepat    data    daya  dokumen  informasi  \
DF      1.0000   1.0000  1.0000  1.0000  3.0000  1.0000   1.0000     1.0000   
IDF     1.6021   1.6021  1.6021  1.6021  1.1249  1.6021   1.6021     1.6021   

     jaringan  kecerdasan  ...  mengolah   mesin  operasi  pembelajaran  \
DF     1.0000      1.0000  ...    1.0000  1.0000   1.0000        1.0000   
IDF    1.6021      1.6021  ...    1.6021  1.6021   1.6021        1.6021   

     pencarian  perangkat  sistem  sumber    teks    temu  
DF      1.0000      2.000   2.000  1.0000  1.0000  1.0000  
IDF     1.6021      1.301   1.301  1.6021  1.6021  1.6021  

[2 rows x 28 columns]


In [7]:
# Menghitung Matriks TF-IDF Manual (TF * IDF)
tfidf_manual_list = []
for baris_tf in tf_list:
    baris_tfidf = {}
    for kata in vocab:
        baris_tfidf[kata] = baris_tf[kata] * idf_dict[kata]
    tfidf_manual_list.append(baris_tfidf)

df_tfidf_manual = pd.DataFrame(tfidf_manual_list, index=[f"D{i}" for i in range(1, len(dokumen) + 1)])
print("=" * 70)
print("MATRIKS TF-IDF HASIL PERHITUNGAN MANUAL")
print("=" * 70)
print(df_tfidf_manual.round(4))

MATRIKS TF-IDF HASIL PERHITUNGAN MANUAL
    algoritma  berbagi  buatan  cepat    data    daya  dokumen  informasi  \
D1     0.0000   0.0000  0.0000  0.000  0.0000  0.1602    0.000      0.000   
D2     0.0000   0.2289  0.0000  0.000  0.1607  0.0000    0.000      0.000   
D3     0.2003   0.0000  0.2003  0.000  0.1406  0.0000    0.000      0.000   
D4     0.0000   0.0000  0.0000  0.178  0.1250  0.0000    0.178      0.178   

    jaringan  kecerdasan  ...  mengolah   mesin  operasi  pembelajaran  \
D1    0.0000      0.0000  ...    0.0000  0.0000   0.1602        0.0000   
D2    0.2289      0.0000  ...    0.0000  0.0000   0.0000        0.0000   
D3    0.0000      0.2003  ...    0.2003  0.2003   0.0000        0.2003   
D4    0.0000      0.0000  ...    0.0000  0.0000   0.0000        0.0000   

    pencarian  perangkat  sistem  sumber   teks   temu  
D1      0.000     0.1301  0.2602  0.1602  0.000  0.000  
D2      0.000     0.1859  0.0000  0.0000  0.000  0.000  
D3      0.000     0.0000  0.0000

In [8]:
# Menghitung Matriks TF-IDF menggunakan Scikit-Learn (TfidfVectorizer)
dokumen_kalimat_bersih = [" ".join(doc) for doc in dokumen_bersih]

vectorizer = TfidfVectorizer(vocabulary=vocab)
matriks_sklearn = vectorizer.fit_transform(dokumen_kalimat_bersih)

df_tfidf_sklearn = pd.DataFrame(
    matriks_sklearn.toarray(),
    columns=vocab,
    index=[f"D{i}" for i in range(1, len(dokumen) + 1)]
)

print("=" * 70)
print("MATRIKS TF-IDF DARI SCIKIT-LEARN")
print("=" * 70)
print(df_tfidf_sklearn.round(4))

MATRIKS TF-IDF DARI SCIKIT-LEARN
    algoritma  berbagi  buatan   cepat    data    daya  dokumen  informasi  \
D1     0.0000   0.0000  0.0000  0.0000  0.0000  0.3206   0.0000     0.0000   
D2     0.0000   0.4207  0.0000  0.0000  0.2685  0.0000   0.0000     0.0000   
D3     0.3674   0.0000  0.3674  0.0000  0.2345  0.0000   0.0000     0.0000   
D4     0.0000   0.0000  0.0000  0.3529  0.2253  0.0000   0.3529     0.3529   

    jaringan  kecerdasan  ...  mengolah   mesin  operasi  pembelajaran  \
D1    0.0000      0.0000  ...    0.0000  0.0000   0.3206        0.0000   
D2    0.4207      0.0000  ...    0.0000  0.0000   0.0000        0.0000   
D3    0.0000      0.3674  ...    0.3674  0.3674   0.0000        0.3674   
D4    0.0000      0.0000  ...    0.0000  0.0000   0.0000        0.0000   

    pencarian  perangkat  sistem  sumber    teks    temu  
D1     0.0000     0.2528  0.5055  0.3206  0.0000  0.0000  
D2     0.0000     0.3317  0.0000  0.0000  0.0000  0.0000  
D3     0.0000     0.0000  0.

In [9]:
# Menampilkan perbandingan term dengan bobot tertinggi di setiap dokumen
print("=" * 75)
print("TERM DENGAN BOBOT TF-IDF TERTINGGI PADA SETIAP DOKUMEN")
print("=" * 75)

for doc_id in df_tfidf_manual.index:
    term_manual = df_tfidf_manual.loc[doc_id].idxmax()
    bobot_manual = df_tfidf_manual.loc[doc_id].max()
    
    term_sklearn = df_tfidf_sklearn.loc[doc_id].idxmax()
    bobot_sklearn = df_tfidf_sklearn.loc[doc_id].max()
    
    print(f"\n{doc_id}:")
    print(f"  - Perhitungan Manual : '{term_manual}' (Bobot: {bobot_manual:.4f})")
    print(f"  - Scikit-Learn       : '{term_sklearn}' (Bobot: {bobot_sklearn:.4f})")

TERM DENGAN BOBOT TF-IDF TERTINGGI PADA SETIAP DOKUMEN

D1:
  - Perhitungan Manual : 'sistem' (Bobot: 0.2602)
  - Scikit-Learn       : 'sistem' (Bobot: 0.5055)

D2:
  - Perhitungan Manual : 'berbagi' (Bobot: 0.2289)
  - Scikit-Learn       : 'berbagi' (Bobot: 0.4207)

D3:
  - Perhitungan Manual : 'algoritma' (Bobot: 0.2003)
  - Scikit-Learn       : 'algoritma' (Bobot: 0.3674)

D4:
  - Perhitungan Manual : 'cepat' (Bobot: 0.1780)
  - Scikit-Learn       : 'cepat' (Bobot: 0.3529)


penerapan metode pembobotan TF-IDF menunjukkan akurasi yang tinggi dalam mengekstraksi kata kunci penting dari sebuah teks. Baik melalui perhitungan matriks secara manual maupun menggunakan pustaka scikit-learn, algoritma TF-IDF terbukti mampu menyoroti kata-kata pembeda yang unik pada setiap dokumen. Hal ini dapat dilihat secara jelas dari term seperti 'sistem', 'berbagi', 'algoritma', dan 'cepat' yang berhasil memperoleh nilai bobot paling dominan karena kata-kata tersebut paling spesifik dalam mendefinisikan topik dari masing-masing dokumen pengujian.